#  GoodWe EV ChargeOps Assistant — Sprint 2
**EV Challenge 2026 | GoodWe Brazil**

Integrantes:
- Brenno Gomes — RM: 570525
- Eduardo Moreira — RM: 569923
- Enzo Stahal — RM: 569001
- Matheus Bruno — RM: 572944

---
### Como usar:
1. Adicione sua `OPENAI_API_KEY` nos **Secrets** do Colab (ícone 🔑 na barra lateral)
2. Execute as células em ordem
3. Use a célula interativa no final para conversar com o chatbot

In [ ]:
!pip install openai --quiet

In [ ]:
import os
import json
from datetime import datetime
from openai import OpenAI

# Configuração segura da API Key via Colab Secret 
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('✅ API Key carregada via Colab Secrets')
except Exception:
    print('⚠️  Colab Secrets indisponível. Certifique-se de que OPENAI_API_KEY está no ambiente.')

In [ ]:
#  System Prompt com contexto GoodWe EV Challenge 2026 
SYSTEM_PROMPT = """
Você é o GoodWe EV ChargeOps Assistant, um assistente operacional inteligente
desenvolvido para o EV Challenge 2026 da GoodWe Brazil.

## SEU PAPEL
Você auxilia síndicos, administradoras de condomínio, moradores e técnicos de
manutenção a gerenciar eletropostos e sessões de recarga de veículos elétricos
em condomínios residenciais equipados com carregadores GoodWe.

## CONTEXTO DO PROBLEMA (EV Challenge 2026)
A GoodWe identificou que condomínios com múltiplos carregadores compartilhados
enfrentam os seguintes desafios:
1. Falta de orquestração de potência entre recargas simultâneas (risco de sobrecarga)
2. Ausência de registro automático de sessões de recarga (kWh, duração, custo)
3. Inexistência de relatórios e faturamento automatizado por morador
4. Dificuldade de comunicação entre moradores e síndico sobre uso e custos
5. Falta de diagnóstico remoto de falhas nos carregadores

## SUAS COMPETÊNCIAS
Você pode auxiliar com:
- **Gestão de sessões**: iniciar, pausar, encerrar e registrar sessões de recarga
- **Orquestração de potência**: regras de balanceamento entre carregadores simultâneos
- **Relatórios**: consumo por morador, custo por período, histórico de recargas
- **Faturamento**: cálculo de custos baseado em kWh consumido e tarifa vigente
- **Regras do condomínio**: horários permitidos, limites de uso, agendamentos
- **Diagnóstico de falhas**: identificar e orientar sobre erros comuns dos carregadores GoodWe
- **Dúvidas técnicas**: especificações dos carregadores GoodWe (séries ESAC, EV chargers)

## PERSONAS QUE VOCÊ ATENDE
- **Síndico/Administrador**: gestão completa, relatórios, cobranças, configuração de regras
- **Morador/Usuário**: consulta de consumo próprio, agendamento, dúvidas de uso
- **Técnico de manutenção**: diagnóstico, erros, especificações técnicas

## COMPORTAMENTO ESPERADO
- Responda sempre em português do Brasil
- Seja objetivo e direto, mas acolhedor
- Para solicitações fora do escopo, redirecione gentilmente ao tema de eletropostos/GoodWe
- Quando não souber algo específico, indique que o usuário consulte o manual técnico GoodWe ou o suporte oficial
- Use linguagem adequada ao perfil: técnica para técnicos, simples para moradores

## EXEMPLOS (few-shot prompting)

Usuário: Quantos kWh o apartamento 42 consumiu em abril?
Assistente: Para consultar o consumo do apartamento 42 em abril, acesse o painel administrativo em Relatórios > Consumo por Unidade > selecione Abril. O sistema exibirá o total de kWh consumidos, custo calculado e número de sessões realizadas. Posso explicar como exportar esse relatório em PDF ou Excel?

Usuário: O carregador está mostrando erro E04, o que faço?
Assistente: O erro E04 nos carregadores GoodWe indica falha de comunicação com a rede. Passos: (1) Verifique o cabo de rede/Wi-Fi; (2) Reinicie o carregador por 10 segundos; (3) Aguarde 2 minutos. Se persistir, abra chamado técnico com código E04 e número de série.
"""

print('✅ System prompt carregado')

In [ ]:
# Classe do Chatbot com memória de contexto nclass GoodWeChargeBot:
    MODEL = 'gpt-4o-mini'
    MAX_TOKENS = 1024
    TEMPERATURE = 0.4

    def __init__(self):
        api_key = os.environ.get('OPENAI_API_KEY')
        if not api_key:
            raise EnvironmentError('OPENAI_API_KEY não encontrada no ambiente.')
        self.client = OpenAI(api_key=api_key)
        self.conversation_history = []
        self.session_id = datetime.now().strftime('%Y%m%d_%H%M%S')
        print(f'✅ Chatbot iniciado | Sessão: {self.session_id}')

    def send_message(self, user_message: str) -> str:
        self.conversation_history.append({'role': 'user', 'content': user_message})
        messages = [{'role': 'system', 'content': SYSTEM_PROMPT}] + self.conversation_history
        response = self.client.chat.completions.create(
            model=self.MODEL,
            messages=messages,
            max_tokens=self.MAX_TOKENS,
            temperature=self.TEMPERATURE,
        )
        assistant_msg = response.choices[0].message.content
        self.conversation_history.append({'role': 'assistant', 'content': assistant_msg})
        return assistant_msg

    def reset(self):
        self.conversation_history = []
        print('🔄 Histórico limpo.')

    def show_history(self):
        for turn in self.conversation_history:
            role = '👤 Você' if turn['role'] == 'user' else '🤖 Assistente'
            print(f'\n{role}:\n{turn["content"]}')

bot = GoodWeChargeBot()

In [ ]:
# Célula interativa — execute múltiplas vezes para conversar 
# Altere a mensagem abaixo e execute a célula (Ctrl+Enter)

mensagem = "Como faço para ver o consumo de energia do meu apartamento?"  # ← edite aqui

print(f'👤 Você: {mensagem}')
print()
resposta = bot.send_message(mensagem)
print(f'🤖 Assistente: {resposta}')

In [ ]:
#  Execução dos 5 casos de teste (Sprint 1) 
test_cases = [
    "Como posso verificar o consumo de energia do meu apartamento no mês passado?",
    "Três moradores estão carregando ao mesmo tempo e a energia caiu. O que aconteceu e como evitar?",
    "Como gero um relatório mensal de consumo por unidade para cobrar cada morador?",
    "O carregador do box 7 está com erro E04. O que isso significa e como resolver?",
    "Posso agendar a recarga para horário de tarifa mais barata? Como configurar?",
]

bot_test = GoodWeChargeBot()  # instância separada para não misturar com a conversa principal

print('=' * 70)
print('EXECUÇÃO DOS CASOS DE TESTE — SPRINT 2')
print('=' * 70)

results = []
for i, pergunta in enumerate(test_cases, 1):
    print(f'\n--- Caso de Teste {i} ---')
    print(f'Pergunta: {pergunta}')
    resposta = bot_test.send_message(pergunta)
    print(f'Resposta: {resposta}')
    results.append({'caso': i, 'pergunta': pergunta, 'resposta': resposta})
    print()

print('\n✅ Todos os casos de teste executados.')

In [ ]:
# Ver histórico completo da sessão 
bot.show_history()